In [ ]:

# Step 1: Import Libraries

import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity


# Step 2: Load Dataset

data = pd.read_csv("./LP IV Updated Datasets/CBOW/twitter_dataset.csv")

print(data.head())

# Step 3: Preprocessing Data

tweets = data["tweet"].astype(str)

tweets = tweets.apply(lambda x: x.lower())

tweets = tweets.apply(
    lambda x: re.sub(r'[^a-z\s]', '', x)
)

# Tokenize Tweets

sentences = tweets.apply(lambda x: x.split()).tolist()


# Step 4: Train CBOW Word2Vec Model

cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=5,
    min_count=2,
    sg=0,
    workers=4,
    epochs=10
)


# Step 5: Convert Tweets into Embeddings

def document_vector(doc):

    words = [w for w in doc if w in cbow_model.wv]

    if len(words) == 0:
        return np.zeros(50)

    return np.mean(cbow_model.wv[words], axis=0)


tweet_vectors = np.array([
    document_vector(doc) for doc in sentences
])


# Step 6: Get User Input

user_input = input("Enter your search tweet: ")

user_input = user_input.lower()

user_input = re.sub(r'[^a-z\s]', '', user_input)

user_words = user_input.split()

user_vector = document_vector(user_words)


# Step 7: Calculate Cosine Similarity

similarity = cosine_similarity(
    [user_vector],
    tweet_vectors
)[0]


# Step 8: Recommend Similar Tweets

top_indices = np.argsort(similarity)[::-1][:5]

print("\nRecommended Tweets:\n")

for i in top_indices:

    print("Tweet:", tweets.iloc[i])

    print("Similarity:", similarity[i])

    print("----------------------")
